# Story 9: Document and Share a Workflow
**Who:** A scientist who is about to publish results computed with an executorlib workflow, and a colleague who wants
to reproduce them - possibly with a different workflow tool.

**What they want:** A machine-readable description of the workflow, and a record of what was actually executed: which
function with which inputs, which resources, how long it took and which SLURM job ran it.

**Why plain SLURM is painful:** The workflow is spread over batch scripts, Python scripts and shell history. What
exactly ran, with which parameters and in which job, has to be reconstructed from log files - if they still exist.

**How executorlib helps:** `export_workflow_filename` writes the workflow graph in the
[Python Workflow Definition](https://github.com/pythonworkflow/python-workflow-definition) JSON format, which other
workflow frameworks can import. The cache directory records every executed call, and `get_cache_data()` turns it into
an audit trail.

This notebook runs on the **login node** of a SLURM cluster.

## Cluster-specific settings
**Adapt the following cell to your cluster.** It collects everything which depends on the cluster rather than on your science. The values below are the ones used to test this notebook on a single-node SLURM test cluster. The submission template is a [Jinja2](https://jinja.palletsprojects.com) template which executorlib renders for every function call: add the lines your cluster requires, for example `#SBATCH --partition=<name>`, `#SBATCH --account=<project>` or `module load ...` commands before `{{command}}`. The [executorlib-cluster-setup](https://executorlib.readthedocs.io/en/latest/2-hpc-cluster.html#setup-with-claude-code) skill for Claude Code can analyse your cluster and generate a tested template.

In [ ]:
# Cluster-specific settings - adapt these values to your cluster.
submission_template = """\
#!/bin/bash
#SBATCH --output=time.out
#SBATCH --job-name={{job_name}}
#SBATCH --chdir={{working_directory}}
#SBATCH --get-user-env=L
#SBATCH --ntasks={{cores}}
{%- if run_time_max %}
#SBATCH --time={{ [1, run_time_max // 60]|max }}
{%- endif %}
{%- if dependency_list %}
#SBATCH --dependency=afterok:{{ dependency_list | join(',') }}
{%- endif %}

{{command}}
"""

## Step 1: Put the functions in a module
To be reusable by others, the functions of a shared workflow belong in an importable Python module rather than in a
notebook. The exported workflow refers to the functions by their module and name. executorlib adds the current
directory to the `PYTHONPATH` of the submitted jobs, so the module is also found on the compute nodes.

In [ ]:
%%writefile story9_workflow.py
import time


def prepare_structure(lattice_constant):
    # Stand-in for building an atomistic structure.
    return {"positions": [lattice_constant * i for i in range(4)], "n_steps": 100}


def run_simulation(positions, n_steps):
    # Stand-in for a simulation.
    time.sleep(1)
    return sum(-1.0 / (1.0 + x) for x in positions) * n_steps / 100


def analyse(energy, lattice_constant):
    # Stand-in for post-processing.
    return {"lattice_constant": lattice_constant, "energy_per_atom": energy / 4}

In [ ]:
from executorlib import get_item_from_future
from story9_workflow import analyse, prepare_structure, run_simulation


def submit_workflow(exe, lattice_constant):
    structure = exe.submit(prepare_structure, lattice_constant=lattice_constant)
    energy = exe.submit(
        run_simulation,
        positions=get_item_from_future(structure, key="positions"),
        n_steps=get_item_from_future(structure, key="n_steps"),
    )
    return exe.submit(analyse, energy=energy, lattice_constant=lattice_constant)

## Step 2: Export the workflow definition
With `export_workflow_filename` the executor records the submitted calls and their connections and writes them to a
JSON file - like `plot_dependency_graph`, **nothing is executed**, the futures return `None`. The workflow graph is the
same for every executor, so the export can be done locally with the `SingleNodeExecutor`, even without cluster access.

In [ ]:
from executorlib import SingleNodeExecutor

workflow_file = "story9_workflow.json"

with SingleNodeExecutor(export_workflow_filename=workflow_file) as exe:
    submit_workflow(exe, lattice_constant=4.0)

In [ ]:
import json

with open(workflow_file) as f:
    workflow_dict = json.load(f)

print(json.dumps(workflow_dict, indent=1))

The JSON file lists the function nodes by module and name, the input values, and the edges which connect an output
(`sourcePort`, for example the `positions` entry of the dictionary returned by `prepare_structure`) to an input
argument (`targetPort`). Together with `story9_workflow.py` it describes the workflow completely, and the
[python-workflow-definition](https://github.com/pythonworkflow/python-workflow-definition) package can load it into
other workflow frameworks like [aiida](https://www.aiida.net), [jobflow](https://materialsproject.github.io/jobflow)
and [pyiron](https://pyiron.org).

## Step 3: Run the workflow on the cluster

In [ ]:
from executorlib import SlurmClusterExecutor

cache_directory = "./story9_cache"

with SlurmClusterExecutor(
    cache_directory=cache_directory,
    resource_dict={"submission_template": submission_template, "cores": 1, "run_time_max": 300},
) as exe:
    result_lst = [submit_workflow(exe, lattice_constant=a).result() for a in [3.9, 4.0]]

result_lst

## Step 4: The cache as audit trail
Each executed call is stored in the cache directory together with its metadata. `get_cache_data()` returns one
dictionary per call with the fields:

| Field | Content |
|---|---|
| `function` | the executed Python function |
| `input_args`, `input_kwargs` | the arguments - inputs provided by another call are stored as a reference to its result |
| `output` / `error` | the return value, or the exception if the call failed |
| `resource_dict` | the resources requested for the call |
| `runtime` | the run time of the function in seconds |
| `queue_id` | the SLURM job ID, for `sacct -j <queue_id>` |
| `filename` | the cache file of the call |

In [ ]:
import pandas
from executorlib import get_cache_data

df = pandas.DataFrame(get_cache_data(cache_directory=cache_directory))
df["function"] = df["function"].apply(lambda fn: f"{fn.__module__}.{fn.__name__}")
df["cores"] = df["resource_dict"].apply(lambda resource_dict: resource_dict["cores"])
df["run_time_max"] = df["resource_dict"].apply(lambda resource_dict: resource_dict.get("run_time_max"))
df[["function", "output", "cores", "run_time_max", "runtime", "queue_id"]].sort_values("queue_id")

Store this table - for example as CSV next to your results - together with the workflow JSON, the module and the
versions of your software. The cache itself uses [cloudpickle](https://github.com/cloudpipe/cloudpickle), whose format
can change between Python and executorlib versions, so it is an audit trail for the current project, not a long-term
archive.

## Clean up

In [ ]:
import glob
import os
import shutil

shutil.rmtree(cache_directory, ignore_errors=True)
for file_name in [workflow_file, "story9_workflow.py"] + glob.glob("__pycache__/story9_workflow.*.pyc"):
    if os.path.exists(file_name):
        os.remove(file_name)
if os.path.isdir("__pycache__") and len(os.listdir("__pycache__")) == 0:
    os.rmdir("__pycache__")

## What you learned
* `export_workflow_filename` writes the workflow graph as Python Workflow Definition JSON without executing it.
* Functions of shared workflows belong in an importable module.
* `get_cache_data()` provides function, inputs, outputs, errors, resources, run time and SLURM job ID of every call.

**Reference:** [dependencies](https://executorlib.readthedocs.io/en/latest/1-single-node.html#dependencies),
[cache](https://executorlib.readthedocs.io/en/latest/1-single-node.html#cache),
[verifying the resource assignment](https://executorlib.readthedocs.io/en/latest/2-hpc-cluster.html#verifying-the-resource-assignment) and the
[Python Workflow Definition](https://github.com/pythonworkflow/python-workflow-definition).